# Lab 6.4 &mdash; Citations Bound to Spans, and Refusing

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 2 &middot; Module 6 &mdash; Agentic RAG**

### What you'll do
- Declare a citation as a <strong>Pydantic</strong> schema the model has to fill
- Pick the parser that actually rejects a bad one &mdash; one of the two does not
- Bind every claim to the exact characters that support it, and drop the ones that cannot be bound
- Refuse when the corpus cannot answer &mdash; structurally, and then in the prompt

> **How this lab works.** You write real LangChain code. Fill every `BLANK`, then run the
> **Self-check** cell under each section &mdash; those check the *objects you built* (a chunked
> `Document`, a Chroma collection, a bound tool, a compiled graph, a parser), so they are
> deterministic and do not depend on the model. Cells marked **Run it for real** put your code in
> front of the sandbox model; that is the part worth watching. The score line is feedback, not a
> grade.

> **Extractive grounding.** Every claim here is a quotation, so the binding is exact
> and a citation is checkable by string comparison. Looser generation needs the
> faithfulness score from Lab 6.5 &mdash; but this is the version you can prove.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-6-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Thinking is off by default here because you will make a lot of calls today;
# pass think=True to any call below to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the corpus (synthetic, self-contained)
# Two short operating documents about the same payments. Read 3.2: the rule and the exception
# that qualifies it are adjacent sentences, which is the whole of Lab 6.1's first lesson. Note
# also what is NOT here -- nothing mentions FX or hedging anywhere, and Lab 6.4 needs that gap.

DOCS = {
    "ops-runbook-v4.md": """## 3.1 Insufficient funds
A payment returned INSUFFICIENT_FUNDS is retried once after 24 hours. If the retry also fails,
notify the client desk. Operations must not fund the account manually.

## 3.2 Limit breaches
Payments above USD 500,000 require Treasury approval before release. This does not apply to
intra-group transfers, which settle same-day without any approval.

## 3.3 Invalid beneficiary details
A payment returned INVALID_IBAN is returned to the originator with code R04. Beneficiary
details are never repaired in-house.

## 3.4 Sanctions review
A payment held for SANCTIONS_REVIEW is decided by Compliance. Operations must not release or
cancel it under any circumstances.
""",
    "escalation-policy-v2.md": """## 1 Approval authority
A duty manager may approve a release up to USD 250,000. Above that figure Treasury approval is
required, and must be recorded against the payment reference.

## 2 Escalation timers
If an approver has not responded within 15 minutes, escalate to the Treasury lead, and after a
further 15 minutes to the head of operations.
""",
}

print(f"{len(DOCS)} documents, {sum(len(d) for d in DOCS.values())} characters")

In [ ]:
# ------------------------------------------------- the embedding model (nothing to fill in)
# The sandbox has no egress, and chromadb's DEFAULT embedding function downloads about 80 MB
# of ONNX model the first time it is called. So this module brings its own: one hashed bucket
# per meaningful word, normalised to unit length. It is arithmetic rather than learning, which
# is the point -- it runs offline, it is deterministic, and you can read every line of it.
#
# What it CAN do: score two texts by the words they share. What it CANNOT do: match meaning
# with no words in common. Lab 6.2 is about living with exactly that.
import re, math, hashlib
from langchain_core.embeddings import Embeddings

STOP = set("""a an the of for is are was were do does did what which who this that these those it
its to in on at by with from about and or not no be been have has had can could should would will
you your we our i me my how why when where there here as if then than so such only just also very
more most some any other""".split())

def content_words(text: str) -> list:
    """The words worth indexing: lower-cased, no punctuation, no stop words."""
    return [w for w in re.findall(r"[a-z0-9_]+", (text or "").lower())
            if w not in STOP and len(w) > 1]


class LabEmbeddings(Embeddings):
    """A tiny embedding model you can read. Same interface as any other LangChain embedding."""

    dim = 1024                      # enough buckets that two different words rarely collide

    def _vector(self, text: str) -> list:
        vec = [0.0] * self.dim
        for word in content_words(text):
            bucket = int(hashlib.sha256(word.encode()).hexdigest()[:8], 16) % self.dim
            vec[bucket] += 1.0
        length = math.sqrt(sum(x * x for x in vec)) or 1.0
        return [x / length for x in vec]      # unit length, so cosine is just a dot product

    def embed_documents(self, texts: list) -> list:
        return [self._vector(t) for t in texts]

    def embed_query(self, text: str) -> list:
        return self._vector(text)


print("embeddings:", LabEmbeddings.dim, "dimensions, offline, deterministic")

In [ ]:
# ------------------------------------------------- carried forward from Lab 6.1 (nothing to fill in)
# Exactly what you built in Lab 6.1: split on headings, index in Chroma, search with a floor.
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_chroma import Chroma

FLOOR = 0.20            # the similarity a chunk must clear to be used at all (Lab 6.1)

def section_chunks() -> list:
    """One Document per '##' section, with the heading kept in the text and in the metadata."""
    splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("##", "section")],
                                          strip_headers=False)
    out = []
    for name, text in DOCS.items():
        for chunk in splitter.split_text(text):
            chunk.metadata["source"] = name
            out.append(chunk)
    return out


_store = None
def store():
    """The Chroma collection, built once, on first use."""
    global _store
    if _store is None:
        chunks = section_chunks()
        _store = Chroma(collection_name="module6-corpus",
                        embedding_function=LabEmbeddings(),
                        persist_directory=os.path.join(WORK, "chroma"),
                        collection_configuration={"hnsw": {"space": "cosine"}})
        # ids derived from the chunk, so re-running this notebook updates instead of duplicating
        _store.add_documents(chunks, ids=[f"{c.metadata['source']}#{c.metadata['section']}"
                                          for c in chunks])
    return _store


def search(query: str, k: int = 4, floor: float = 0.0, where: dict | None = None) -> list:
    """Top-k from the store as plain dicts, with anything below `floor` dropped."""
    hits = store().similarity_search_with_score(query, k=k, filter=where)
    out = []
    for doc, distance in hits:
        similarity = 1.0 - distance         # cosine space: 1.0 identical, 0.0 nothing in common
        if similarity >= floor:
            out.append({"score": round(similarity, 3), "text": doc.page_content,
                        "source": doc.metadata["source"], "section": doc.metadata["section"]})
    return out


print(f"index ready: {len(store().get()['ids'])} chunks")

## Concept

Two behaviours a regulated client will ask about, and both have to be **mechanisms** rather than
requests, because a request is something the model can decline to honour on any given run.

- **Citation** &mdash; not &ldquo;here are the documents that were in context&rdquo;, but *this claim came
  from these characters of that section*.
- **Refusal** &mdash; not &ldquo;the model decided it did not know&rdquo;, but *nothing cleared the floor, so
  there is nothing to answer from*.

The schema and the parser are how you state the first one to the model. The floor from Lab 6.1 is
how you get the second without asking for it.

## Section 1 &mdash; Declare what a citation is

A Pydantic model is two things at once: the shape you validate against, and &mdash; through the
parser's format instructions &mdash; the description the model reads. The `Field` descriptions are
sent to the model verbatim, so they are instructions, not comments.

In [ ]:
from pydantic import BaseModel, Field

class Citation(BaseModel):
    """One claim, bound to the text that supports it."""

    claim: str = Field(description="BLANK")
    # TODO (claim): one line the model can follow. What is a claim here -- a whole answer, or
    # a single assertion that one span of one section can support on its own?

    quote: str = Field(description="BLANK")
    # TODO (quote): this is what gets matched against the source, character for character.
    # Say that it must be copied EXACTLY -- use the word "exactly" -- and never paraphrased.

    source: str = Field(description="the file the quote came from, e.g. 'ops-runbook-v4.md'")
    section: str = Field(description="the section heading the quote came from, e.g. '3.2 Limit breaches'")

In [ ]:
# --- Self-check: Section 1   (the schema object -- no model)
def _rejects(fn) -> bool:
    """True if fn() refused its input. NameError is re-raised so a blank still prints [TODO]."""
    try:
        fn()
    except NameError:
        raise
    except Exception:
        return True
    return False

def _field_desc(name: str) -> str:
    d = (Citation.model_fields[name].description or "").strip()
    if d == "BLANK" or not d:
        raise NameError(f"{name} still has the placeholder description")
    return d

check("the schema declares all four fields",
      lambda: set(Citation.model_fields) == {"claim", "quote", "source", "section"})
check("every field carries a description the model will be shown",
      lambda: all(_field_desc(f) for f in Citation.model_fields))
check("the claim description says a claim is ONE assertion",
      lambda: any(w in _field_desc("claim").lower() for w in ("one ", "single", "a single")),
      "a citation attached to a whole paragraph cannot be checked against a span")
check("the quote description demands an exact copy",
      lambda: "exact" in _field_desc("quote").lower(),
      "a paraphrased quote cannot be found in the source, so it cannot be verified")
check("a well-formed citation validates",
      lambda: Citation(claim="c", quote="q", source="s", section="3.2").quote == "q")
check("and a citation with no section does not",
      lambda: _rejects(lambda: Citation(claim="c", quote="q", source="s")))

## Section 2 &mdash; The parser that actually rejects a bad one

LangChain gives you two parsers that both take `pydantic_object=`. Only one of them validates
against it. The other writes the format instructions and then hands you back whatever JSON the
model produced, missing fields and all &mdash; which looks identical right up to the moment
something downstream reads `citation["section"]`.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser, JsonOutputParser
from langchain_core.exceptions import OutputParserException

def citation_parser():
    """The parser used on model output. It must REJECT a citation that is missing a field."""
    # TODO: PydanticOutputParser or JsonOutputParser? The self-check below is the experiment --
    # try the other one and read what it does with `MISSING_SECTION`.
    return BLANK(pydantic_object=Citation)


GOOD_JSON = json.dumps({"claim": "Payments above USD 500,000 require Treasury approval",
                        "quote": "Payments above USD 500,000 require Treasury approval",
                        "source": "ops-runbook-v4.md", "section": "3.2 Limit breaches"})
MISSING_SECTION = json.dumps({"claim": "Payments above USD 500,000 require Treasury approval",
                              "quote": "Payments above USD 500,000 require Treasury approval",
                              "source": "ops-runbook-v4.md"})

In [ ]:
# --- Self-check: Section 2   (parser objects, on fixed strings -- no model)
check("a well-formed citation parses into a Citation object",
      lambda: isinstance(citation_parser().parse(GOOD_JSON), Citation),
      "a parser that hands back a dict has validated nothing")
check("A CITATION MISSING ITS SECTION IS REJECTED",
      lambda: _rejects(lambda: citation_parser().parse(MISSING_SECTION)),
      "JsonOutputParser(pydantic_object=...) accepts this quietly -- try it and see")
check("the format instructions carry your field descriptions to the model",
      lambda: "quote" in citation_parser().get_format_instructions()
              and "section" in citation_parser().get_format_instructions())
check("the instructions are worth sending -- they are the schema in words",
      lambda: len(citation_parser().get_format_instructions()) > 200)

## Section 3 &mdash; No span, no claim

A citation that names a document proves nothing: the document was in the context whatever the
model wrote. A *span* is checkable &mdash; you can slice the source and compare.

`compose` is the control: a claim whose quote cannot be found in what was retrieved is
**dropped**, not flagged. Anything less and you have added a field, not a control.

In [ ]:
def normalise(text: str) -> str:
    return " ".join((text or "").split()).lower()


def find_span(quote: str, chunk: dict):
    """The (start, end) character range in the chunk that contains this quote, or None."""
    hay, needle = normalise(chunk["text"]), normalise(quote)
    i = hay.find(needle)
    return (i, i + len(needle)) if i >= 0 and needle else None


def bind(citation: Citation, results: list):
    """Attach the first retrieved chunk that actually contains this citation's quote."""
    for r in results:
        span = find_span(citation.quote, r)
        if span:
            return {"claim": citation.claim, "source": r["source"],
                    "section": r["section"], "span": span}
    return None


def compose(citations: list, results: list) -> dict:
    """Keep only the claims that can name their source. Report what was dropped."""
    bound = [(c, bind(c, results)) for c in citations]
    kept = [b for c, b in bound if b is not None]
    dropped = [c.claim for c, b in bound if b is None]
    return {"claims": [b["claim"] for b in kept], "dropped": dropped,
            "citations": [f"{b['source']}#{b['section']} [{b['span'][0]}:{b['span'][1]}]"
                          for b in kept]}

In [ ]:
# --- Self-check: Section 3   (span binding over real retrievals -- no model)
LIMIT_Q = "limit breach approval above USD 500,000"
LIMIT_HITS = search(LIMIT_Q, k=3)

SUPPORTED = Citation(claim="Large payments need Treasury approval",
                     quote="Payments above USD 500,000 require Treasury approval before release",
                     source="ops-runbook-v4.md", section="3.2 Limit breaches")
EXCEPTION = Citation(claim="Intra-group transfers are exempt",
                     quote="This does not apply to intra-group transfers",
                     source="ops-runbook-v4.md", section="3.2 Limit breaches")
INVENTED  = Citation(claim="The duty manager may release it",
                     quote="Payments above USD 500,000 may be released by the duty manager",
                     source="ops-runbook-v4.md", section="3.2 Limit breaches")

check("a supported citation finds its span",
      lambda: bind(SUPPORTED, LIMIT_HITS) is not None)
check("and the span points into the right section",
      lambda: bind(SUPPORTED, LIMIT_HITS)["section"].startswith("3.2"))
check("the span is a real character range you can slice",
      lambda: normalise(SUPPORTED.quote) in
              normalise(next(r["text"] for r in LIMIT_HITS
                             if r["section"] == bind(SUPPORTED, LIMIT_HITS)["section"])),
      "an auditor follows one link; the check is a string comparison, not a judgement")
check("AN INVENTED QUOTE BINDS TO NOTHING",
      lambda: bind(INVENTED, LIMIT_HITS) is None,
      "it is plausible, it is about the retrieved topic, and it is not in the text")
check("whitespace differences do not break a real citation",
      lambda: bind(Citation(claim="c", quote="Payments above USD 500,000\n   require Treasury",
                            source="s", section="3.2"), LIMIT_HITS) is not None)
check("compose keeps the two supported claims and drops the invented one",
      lambda: compose([SUPPORTED, EXCEPTION, INVENTED], LIMIT_HITS)["dropped"]
              == [INVENTED.claim])
check("every surviving claim has a citation with a section and a range",
      lambda: all("#3.2" in c and "[" in c
                  for c in compose([SUPPORTED, EXCEPTION], LIMIT_HITS)["citations"]))
check("the exception survives alongside the rule, because Lab 6.1 chunked them together",
      lambda: EXCEPTION.claim in compose([SUPPORTED, EXCEPTION], LIMIT_HITS)["claims"],
      "chunk them apart and this claim becomes uncitable, so this control would delete it")

## Section 4 &mdash; Refuse, usefully

There are two refusals here and they are not alternatives.

The **structural** one needs no co-operation: the floor from Lab 6.1 empties the result set, so
there is no context and nothing to be wrong from.

The **prompted** one matters when there *is* context and it still does not answer the question.
Measured on this sandbox: with an explicit refusal clause the model flagged it 3/3; without one,
0/3. The clause is not decoration.

In [ ]:
def refusal_clause() -> str:
    """The sentence in the system prompt that makes refusing an available answer."""
    # TODO: write it. It has to (a) tell the model to answer only from the context, (b) tell it
    # what to do when the context does not cover the question, and (c) make the refusal
    # MACHINE-READABLE by requiring the exact token INSUFFICIENT_CONTEXT in that case.
    return "BLANK"


def refused(reply: str) -> bool:
    """Did the model refuse? Read the token, not the tone."""
    return "INSUFFICIENT_CONTEXT" in (reply or "")


def respond(question: str, citations=None, floor: float = FLOOR) -> dict:
    """Answer from the corpus, or refuse and say what was missing. No model involved."""
    results = search(question, k=3, floor=floor)
    if not results:
        nearest = search(question, k=1)          # what we would have used, had we allowed it
        near = nearest[0]["section"] if nearest else "nothing"
        topic = ", ".join(sorted(set(content_words(question)))[:4])
        return {"answered": False, "citations": [],
                "why": f"nothing in the corpus clears the bar for [{topic}]; "
                       f"the closest section is {near}"}
    out = compose(citations or [], results)
    if not out["claims"]:
        return {"answered": False, "citations": [],
                "why": f"retrieved {len(results)} section(s) but no claim could name a span"}
    return {"answered": True, "citations": out["citations"],
            "claims": out["claims"], "dropped": out["dropped"]}

In [ ]:
# --- Self-check: Section 4   (the clause as a string, the floor as a mechanism -- no model)
FX_Q = "what is the FX hedging policy for JPY exposure"

def _clause() -> str:
    c = refusal_clause().strip()
    if c == "BLANK" or not c:
        raise NameError("refusal_clause() is still the placeholder")
    return c

check("the clause is a real instruction, not a word",
      lambda: len(_clause()) > 60)
check("it confines the model to the context",
      lambda: "context" in _clause().lower())
check("and it names the token your code reads",
      lambda: "INSUFFICIENT_CONTEXT" in _clause(),
      "'say you do not know' is unreadable by machine -- refused() has to detect something exact")
check("refused() detects that token and not a mood",
      lambda: refused("INSUFFICIENT_CONTEXT nothing here covers FX") is True
              and refused("I'm not really sure about that") is False)
check("an answerable question is answered, with a citation",
      lambda: respond(LIMIT_Q, citations=[SUPPORTED])["answered"] is True
              and len(respond(LIMIT_Q, citations=[SUPPORTED])["citations"]) == 1)
check("a question the corpus cannot answer is REFUSED before any model sees it",
      lambda: respond(FX_Q, citations=[SUPPORTED])["answered"] is False,
      "the floor emptied the result set -- there is no context to be wrong from")
check("the refusal names what was searched for and what is nearby",
      lambda: "hedging" in respond(FX_Q)["why"] and "closest section" in respond(FX_Q)["why"],
      "that sentence is a work item for whoever owns the corpus")
check("retrieving something and supporting nothing also refuses",
      lambda: respond(LIMIT_Q, citations=[INVENTED])["answered"] is False,
      "the second gate: results cleared the floor, and still no claim could name a span")

## Run it for real &mdash; part 1: ask the model to cite

The schema's format instructions go into the prompt, the model answers, your parser validates it,
and `bind` checks the quote against the text it claims to come from.

In [ ]:
if llm_ready():
    def _cited_answer():
        parser = citation_parser()
        results = search(LIMIT_Q, k=3)
        context = "\n\n".join(f"[{r['source']} #{r['section']}]\n{r['text']}" for r in results)
        raw = ask(f"CONTEXT:\n{context}\n\nQuestion: What approval does a payment above "
                  f"USD 500,000 need?\n\n{parser.get_format_instructions()}",
                  system="Reply with the JSON object only.")
        print("  raw reply:", raw.strip()[:200].replace("\n", " "))
        try:
            cited = parser.parse(raw)
        except Exception as exc:
            print(f"  parser REJECTED it: {type(exc).__name__} -- and that is the parser working")
            return
        bound = bind(cited, results)
        print(f"  claim   : {cited.claim[:90]}")
        print(f"  quote   : {cited.quote[:90]}")
        print(f"  binds to: {bound['section'] + ' ' + str(bound['span']) if bound else 'NOTHING'}")
    guard(_cited_answer)

## Run it for real &mdash; part 2: the refusal clause is the whole difference

The same unanswerable question, three times: with no context at all, with the four nearest chunks
and no refusal clause, and with the same four chunks and your clause.

In [ ]:
if llm_ready():
    def _refusals():
        nearest = search(FX_Q, k=4)                    # deliberately NO floor -- the wrong context
        context = "\n".join(f"- [{r['section']}] {r['text'][:160]}" for r in nearest)
        arms = [
            ("floor applied: no context",  "(no documents were retrieved)", _clause()),
            ("context, no refusal clause", context, "Be brief."),
            ("context, refusal clause",    context, _clause()),
        ]
        for label, ctx, system in arms:
            reply = ask(f"CONTEXT:\n{ctx}\n\nQuestion: {FX_Q}", system=system)
            print(f"  [{label:28}] refused={refused(reply)}")
            print(f"      {reply.strip()[:180]}")
            print()
    guard(_refusals)

### Read it

**The citation.** If the parser rejected the reply, that is the parser doing its job &mdash; note that
`JsonOutputParser` would have accepted the same string and handed you a dict with a missing key.
If it parsed and then bound to nothing, look at the quote: it will be a *paraphrase*. That is why
the `quote` description says &ldquo;exactly&rdquo;, and it is the commonest way a citation stops being
checkable.

**The refusals.** The first arm is structural: no context, nothing to be wrong from, and it does not
depend on the model behaving. The third arm is the clause working. The middle arm is the one to
stare at &mdash; four chunks about payment limits in front of a question about FX, and nothing but
the model's own judgement between you and an answer stitched out of the nearest available prose.

Sometimes the middle arm refuses anyway. That is worth noticing and not worth relying on: you
cannot put &ldquo;the model was sensible&rdquo; in a control document, and it is not the same sentence in
the next model version.

In [ ]:
score()

## Your turn

1. Swap `citation_parser()` to `JsonOutputParser` and re-run the Section 2 checks. Then find the
   line of code downstream that would have crashed in production instead.
2. Extractive grounding is the strictest kind and the least fluent. Let the model paraphrase, then
   decide how you would still bind a claim to a span &mdash; and what you lose when the match stops
   being exact.
3. `respond` drops unsupported claims silently. Log them instead, and after a day of traffic read
   the log: the claims a model keeps trying to make and cannot support are a map of what your
   corpus is missing.